# 06 - Analise Estatistica da Segmentacao Binarizada

Carrega a base analitica da segmentacao binarizada a partir do SQLite, filtrando a execucao escolhida em `config.toml`, persiste os resultados estatisticos no banco e exibe uma leitura inicial para comparacao entre estrategias e modelos.


In [1]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import (
    build_and_persist_analysis_segmentacao_binarizada_interacao_tag_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_intervalo_confianca,
    build_and_persist_analysis_segmentacao_binarizada_resumo_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_resumo_modelo_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_resumo_tag,
    build_and_persist_analysis_segmentacao_binarizada_testes_estrategia,
    build_and_persist_analysis_segmentacao_binarizada_testes_tag_estrategia,
    build_binarized_metrics_dataframe,
)
from src.config import SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO
from src.repositories import (
    AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository,
    AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaResumoTagRepository,
    AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaTesteTagEstrategiaRepository,
    ImagemRepository,
)


/home/victor/Desktop/projeto-bufalos/worktrees/analise-estatistica-segmentacao-bruta/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregamento da base analitica em memoria

Le a avaliacao persistida no SQLite e monta a base analitica por `imagem + modelo + estrategia_binarizacao` apenas para a execucao escolhida em `config.toml`.


In [2]:
imagem_repository = ImagemRepository()
# Docs: docs/decisoes-tecnicas/analise-da-segmentacao-binarizada-por-execucao-fixa.md
df_base = build_binarized_metrics_dataframe(
    imagem_repository.list(),
    execucao_escolhida=SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO,
)

print(f'Execucao escolhida para a analise binarizada: {SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO}')
print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes presentes na base: {sorted(df_base["execucao"].unique().tolist())}')
print(f'Estrategias de binarizacao: {df_base["estrategia_binarizacao"].nunique()}')
print(f'Tags binarias disponiveis: {len([column for column in df_base.columns if column.startswith("tag_")])}')

df_base.head()


Execucao escolhida para a analise binarizada: 1
Registros na base analitica: 43344
Imagens: 387
Modelos: 14
Execucoes presentes na base: [1]
Estrategias de binarizacao: 8
Tags binarias disponiveis: 6


,nome_arquivo,fazenda,peso,modelo,execucao,estrategia_binarizacao,iou,precision,recall,area,...,tags_sem_ok,num_tags_problema,tem_tag_problema,grupo_dificuldade,tag_ok,tag_multi_bufalos,tag_cortado,tag_angulo_extremo,tag_baixo_contraste,tag_ocluido
0,0266d87e-faec-497f-8e14-949a5ab6afa3,Manezinho,78.0,birefnet-cod,1,GaussianaOpeningAlta,0.925964,0.982280,0.941694,396787.0,...,,0,False,ok,True,False,False,False,False,False
1,0266d87e-faec-497f-8e14-949a5ab6afa3,Manezinho,78.0,birefnet-cod,1,GaussianaOpeningBaixa,0.930140,0.976852,0.951103,402978.0,...,,0,False,ok,True,False,False,False,False,False
2,0266d87e-faec-497f-8e14-949a5ab6afa3,Manezinho,78.0,birefnet-cod,1,HistereseClosingAlta,0.929571,0.977803,0.949610,401954.0,...,,0,False,ok,True,False,False,False,False,False
3,0266d87e-faec-497f-8e14-949a5ab6afa3,Manezinho,78.0,birefnet-cod,1,HistereseClosingBaixa,0.931319,0.971450,0.957527,407956.0,...,,0,False,ok,True,False,False,False,False,False
4,0266d87e-faec-497f-8e14-949a5ab6afa3,Manezinho,78.0,birefnet-cod,1,LimiarFixoAlta,0.923759,0.984115,0.937742,394385.0,...,,0,False,ok,True,False,False,False,False,False


## Persistencia da camada analitica

Calcula os resumos descritivos, intervalos de confianca, comparacoes entre estrategias, testes por tag e interacoes `estrategia x tag`, persistindo tudo no SQLite. As analises entre execucoes ficam fora deste notebook por decisao tecnica.


In [3]:
resumo_estrategia_repository = AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository()
resumo_modelo_estrategia_repository = AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository()
resumo_tag_repository = AnaliseSegmentacaoBinarizadaResumoTagRepository()
intervalo_confianca_repository = AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository()
teste_estrategia_repository = AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository()
teste_tag_estrategia_repository = AnaliseSegmentacaoBinarizadaTesteTagEstrategiaRepository()
interacao_tag_estrategia_repository = AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository()

df_resumo_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_estrategia(
    df_base=df_base,
    repository=resumo_estrategia_repository,
)
df_resumo_modelo_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_modelo_estrategia(
    df_base=df_base,
    repository=resumo_modelo_estrategia_repository,
)
df_resumo_tag, _ = build_and_persist_analysis_segmentacao_binarizada_resumo_tag(
    df_base=df_base,
    repository=resumo_tag_repository,
)
df_intervalo_confianca, _ = build_and_persist_analysis_segmentacao_binarizada_intervalo_confianca(
    df_base=df_base,
    repository=intervalo_confianca_repository,
)
df_testes_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_testes_estrategia(
    df_base=df_base,
    repository=teste_estrategia_repository,
)
df_testes_tag_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_testes_tag_estrategia(
    df_base=df_base,
    repository=teste_tag_estrategia_repository,
)
df_interacoes_tag_estrategia, _ = build_and_persist_analysis_segmentacao_binarizada_interacao_tag_estrategia(
    df_base=df_base,
    repository=interacao_tag_estrategia_repository,
)

print(f'Registros no resumo por estrategia: {len(df_resumo_estrategia)}')
print(f'Registros no resumo por modelo + estrategia: {len(df_resumo_modelo_estrategia)}')
print(f'Registros no resumo por tag: {len(df_resumo_tag)}')
print(f'Registros nos intervalos de confianca: {len(df_intervalo_confianca)}')
print(f'Registros nos testes entre estrategias: {len(df_testes_estrategia)}')
print(f'Registros nos testes por tag: {len(df_testes_tag_estrategia)}')
print(f'Registros nas interacoes estrategia x tag: {len(df_interacoes_tag_estrategia)}')


Registros no resumo por estrategia: 24
Registros no resumo por modelo + estrategia: 336
Registros no resumo por tag: 4032
Registros nos intervalos de confianca: 672
Registros nos testes entre estrategias: 1305
Registros nos testes por tag: 162
Registros nas interacoes estrategia x tag: 144


## Leitura inicial dos resultados

Mostra uma visao compacta das tabelas persistidas para orientar a etapa visual do notebook 07.


In [4]:
pd.pivot_table(
    df_resumo_estrategia,
    index='estrategia_binarizacao',
    columns='metric_name',
    values=['mean', 'median'],
)


mean                        median            \
metric_name                  iou precision    recall       iou precision   
estrategia_binarizacao                                                     
GaussianaOpeningAlta    0.901688  0.948601  0.922651  0.964985  0.993282   
GaussianaOpeningBaixa   0.906866  0.943035  0.934514  0.969013  0.988338   
HistereseClosingAlta    0.906608  0.943729  0.932816  0.968930  0.989326   
HistereseClosingBaixa   0.907508  0.936823  0.942262  0.967973  0.982852   
LimiarFixoAlta          0.898515  0.950161  0.917656  0.963844  0.994409   
LimiarFixoBaixa         0.907185  0.940637  0.937337  0.968941  0.986440   
OtsuOpeningAlta         0.914438  0.949876  0.939945  0.967684  0.991468   
OtsuOpeningBaixa        0.918134  0.943965  0.952133  0.968794  0.989160   

                                  
metric_name               recall  
estrategia_binarizacao            
GaussianaOpeningAlta    0.974595  
GaussianaOpeningBaixa   0.983732  
HistereseClosingAlta    0.982567  
HistereseClosingBaixa   0.988553  
LimiarFixoAlta          0.972347  
LimiarFixoBaixa         0.985649  
OtsuOpeningAlta         0.978946  
OtsuOpeningBaixa        0.982653

In [5]:
df_resumo_modelo_estrategia.sort_values(['metric_name', 'modelo', 'mean'], ascending=[True, True, False]).head(24)


,modelo,estrategia_binarizacao,metric_name,count,mean,median,std,min,max,q1,q3,iqr,higher_is_better
2,birefnet-cod,HistereseClosingAlta,iou,387,0.962378,0.972456,0.036522,0.677406,0.988315,0.963995,0.977832,0.013836,True
1,birefnet-cod,GaussianaOpeningBaixa,iou,387,0.962311,0.972487,0.036918,0.673581,0.988271,0.964189,0.977803,0.013613,True
7,birefnet-cod,OtsuOpeningBaixa,iou,387,0.962272,0.972347,0.036935,0.673846,0.988608,0.963908,0.977860,0.013952,True
5,birefnet-cod,LimiarFixoBaixa,iou,387,0.962236,0.972190,0.036883,0.673523,0.987718,0.963980,0.977580,0.013600,True
6,birefnet-cod,OtsuOpeningAlta,iou,387,0.961982,0.972058,0.036996,0.674186,0.989316,0.963419,0.977970,0.014551,True
3,birefnet-cod,HistereseClosingBaixa,iou,387,0.961891,0.971811,0.036647,0.673860,0.986750,0.963843,0.976991,0.013148,True
4,birefnet-cod,LimiarFixoAlta,iou,387,0.960926,0.970844,0.037067,0.674507,0.990231,0.961430,0.977319,0.015889,True
0,birefnet-cod,GaussianaOpeningAlta,iou,387,0.960651,0.970598,0.037308,0.674613,0.989825,0.960692,0.977455,0.016762,True
11,birefnet-dis,HistereseClosingBaixa,iou,387,0.928307,0.977215,0.147323,0.145072,0.994906,0.963458,0.984061,0.020603,True
13,birefnet-dis,LimiarFixoBaixa,iou,387,0.928172,0.977030,0.146633,0.145657,0.994663,0.962925,0.983883,0.020958,True


In [6]:
df_intervalo_confianca.sort_values(['metric_name', 'statistic_name', 'modelo', 'estrategia_binarizacao']).head(24)


,modelo,estrategia_binarizacao,metric_name,statistic_name,count,estimate,ci_low,ci_high,confidence_level,n_resamples,higher_is_better
0,birefnet-cod,GaussianaOpeningAlta,iou,mean,387,0.960651,0.956792,0.964075,0.95,1000,True
6,birefnet-cod,GaussianaOpeningBaixa,iou,mean,387,0.962311,0.958553,0.965688,0.95,1000,True
12,birefnet-cod,HistereseClosingAlta,iou,mean,387,0.962378,0.958180,0.965698,0.95,1000,True
18,birefnet-cod,HistereseClosingBaixa,iou,mean,387,0.961891,0.958374,0.965327,0.95,1000,True
24,birefnet-cod,LimiarFixoAlta,iou,mean,387,0.960926,0.956882,0.964316,0.95,1000,True
30,birefnet-cod,LimiarFixoBaixa,iou,mean,387,0.962236,0.958521,0.965825,0.95,1000,True
36,birefnet-cod,OtsuOpeningAlta,iou,mean,387,0.961982,0.958319,0.965345,0.95,1000,True
42,birefnet-cod,OtsuOpeningBaixa,iou,mean,387,0.962272,0.958350,0.965610,0.95,1000,True
48,birefnet-dis,GaussianaOpeningAlta,iou,mean,387,0.924801,0.910927,0.938414,0.95,1000,True
54,birefnet-dis,GaussianaOpeningBaixa,iou,mean,387,0.927981,0.913506,0.942005,0.95,1000,True


In [7]:
if df_testes_estrategia.empty:
    print('Nao ha estrategias suficientes no banco atual para executar comparacoes estatisticas.')
else:
    df_testes_estrategia.sort_values(['metric_name', 'modelo_origem', 'comparison_scope', 'p_value_adjusted']).head(30)


In [8]:
df_testes_tag_estrategia.sort_values(['metric_name', 'estrategia_binarizacao', 'tag_name', 'comparison_scope', 'p_value_adjusted']).head(30)


,metric_name,tag_name,comparison_scope,estrategia_binarizacao,test_name,n_group_a,n_group_b,statistic,p_value,p_value_adjusted,effect_size,effect_size_label,mean_com_tag,mean_sem_tag,median_com_tag,median_sem_tag,delta_mean,delta_median
28,iou,tag_angulo_extremo,por_modelo,GaussianaOpeningAlta,mann_whitney_u,644,4774,1650593.0,2.346536e-03,1.877229e-02,0.073746,negligible,0.899128,0.902034,0.966233,0.964771,-0.002906,0.001463
37,iou,tag_baixo_contraste,por_modelo,GaussianaOpeningAlta,mann_whitney_u,1442,3976,2365777.0,7.240781e-23,1.448156e-22,-0.174737,small,0.881993,0.908831,0.954003,0.966888,-0.026839,-0.012885
19,iou,tag_cortado,por_modelo,GaussianaOpeningAlta,mann_whitney_u,350,5068,949548.0,2.686189e-02,2.148951e-01,0.070637,negligible,0.902628,0.901623,0.967356,0.964777,0.001004,0.002579
10,iou,tag_multi_bufalos,por_modelo,GaussianaOpeningAlta,mann_whitney_u,840,4578,1516346.0,1.797235e-22,3.594469e-22,-0.211370,small,0.880817,0.905518,0.951981,0.966249,-0.024700,-0.014268
46,iou,tag_ocluido,por_modelo,GaussianaOpeningAlta,mann_whitney_u,154,5264,207471.0,4.595653e-25,3.676522e-24,-0.488140,large,0.847786,0.903265,0.910327,0.965529,-0.055479,-0.055202
1,iou,tag_ok,por_modelo,GaussianaOpeningAlta,mann_whitney_u,3458,1960,3840145.0,3.418238e-16,6.836477e-16,0.133174,negligible,0.908958,0.888862,0.967074,0.958216,0.020096,0.008858
29,iou,tag_angulo_extremo,por_modelo,GaussianaOpeningBaixa,mann_whitney_u,644,4774,1596834.0,1.096690e-01,4.065768e-01,0.038775,negligible,0.903355,0.907340,0.968306,0.969063,-0.003984,-0.000757
38,iou,tag_baixo_contraste,por_modelo,GaussianaOpeningBaixa,mann_whitney_u,1442,3976,2295665.0,3.166151e-29,2.532921e-28,-0.199195,small,0.885747,0.914525,0.958042,0.971018,-0.028778,-0.012977
20,iou,tag_cortado,por_modelo,GaussianaOpeningBaixa,mann_whitney_u,350,5068,922475.0,2.087716e-01,9.073037e-01,0.040112,negligible,0.910035,0.906647,0.969837,0.968997,0.003388,0.000840
11,iou,tag_multi_bufalos,por_modelo,GaussianaOpeningBaixa,mann_whitney_u,840,4578,1435703.0,1.471161e-31,8.826967e-31,-0.253311,small,0.883373,0.911177,0.952130,0.970455,-0.027803,-0.018325


In [9]:
df_interacoes_tag_estrategia.sort_values(['metric_name', 'estrategia_binarizacao', 'adjusted_delta_mean']).head(30)


,estrategia_binarizacao,tag_name,metric_name,count_com_tag,count_sem_tag,mean_com_tag,mean_sem_tag,median_com_tag,median_sem_tag,delta_mean,delta_median,relative_delta_mean,adjusted_delta_mean,adjusted_delta_median,impact_direction,higher_is_better
5,GaussianaOpeningAlta,tag_ocluido,iou,154,5264,0.847786,0.903265,0.910327,0.965529,-0.055479,-0.055202,-0.061421,-0.055479,-0.055202,piora,True
4,GaussianaOpeningAlta,tag_baixo_contraste,iou,1442,3976,0.881993,0.908831,0.954003,0.966888,-0.026839,-0.012885,-0.029531,-0.026839,-0.012885,piora,True
1,GaussianaOpeningAlta,tag_multi_bufalos,iou,840,4578,0.880817,0.905518,0.951981,0.966249,-0.024700,-0.014268,-0.027278,-0.024700,-0.014268,piora,True
3,GaussianaOpeningAlta,tag_angulo_extremo,iou,644,4774,0.899128,0.902034,0.966233,0.964771,-0.002906,0.001463,-0.003221,-0.002906,0.001463,piora,True
2,GaussianaOpeningAlta,tag_cortado,iou,350,5068,0.902628,0.901623,0.967356,0.964777,0.001004,0.002579,0.001114,0.001004,0.002579,melhora,True
0,GaussianaOpeningAlta,tag_ok,iou,3458,1960,0.908958,0.888862,0.967074,0.958216,0.020096,0.008858,0.022609,0.020096,0.008858,melhora,True
11,GaussianaOpeningBaixa,tag_ocluido,iou,154,5264,0.853248,0.908435,0.919847,0.969467,-0.055187,-0.049620,-0.060749,-0.055187,-0.049620,piora,True
10,GaussianaOpeningBaixa,tag_baixo_contraste,iou,1442,3976,0.885747,0.914525,0.958042,0.971018,-0.028778,-0.012977,-0.031468,-0.028778,-0.012977,piora,True
7,GaussianaOpeningBaixa,tag_multi_bufalos,iou,840,4578,0.883373,0.911177,0.952130,0.970455,-0.027803,-0.018325,-0.030514,-0.027803,-0.018325,piora,True
9,GaussianaOpeningBaixa,tag_angulo_extremo,iou,644,4774,0.903355,0.907340,0.968306,0.969063,-0.003984,-0.000757,-0.004391,-0.003984,-0.000757,piora,True
